# Facial Expression Recognition - FER2013

**Dataset**: FER2013 from Kaggle  
**Task**: Classify 7 emotions from 48x48 grayscale face images  
**Emotions**: Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral

---

## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Settings
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Load Dataset

In [ ]:
# Load FER2013 dataset
# Download from: https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge

train_df = pd.read_csv('/kaggle/input/fer2013/fer2013.csv')

print(f"Dataset shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
train_df.head()

## 3. Explore Dataset

In [ ]:
# Emotion labels mapping
emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Check data distribution
print("Dataset split by Usage:")
print(train_df['Usage'].value_counts())

print("\nEmotion distribution:")
print(train_df['emotion'].value_counts().sort_index())

# Visualize emotion distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
train_df['emotion'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(range(7), emotions, rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
train_df['Usage'].value_counts().plot(kind='bar', color='lightcoral')
plt.title('Dataset Split')
plt.xlabel('Usage')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Visualize Sample Images

In [ ]:
# Display sample images for each emotion
def show_sample_images():
    plt.figure(figsize=(14, 8))
    
    for i, emotion_label in enumerate(emotions):
        # Get one sample of this emotion
        sample = train_df[train_df['emotion'] == i].iloc[0]
        
        # Convert pixel string to image
        pixels = np.array([int(p) for p in sample['pixels'].split()]).reshape(48, 48)
        
        # Plot
        plt.subplot(2, 4, i+1)
        plt.imshow(pixels, cmap='gray')
        plt.title(f'{emotion_label} (Label: {i})', fontsize=12, fontweight='bold')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

show_sample_images()

## 5. Data Preprocessing Function

In [ ]:
def prepare_data(df):
    """
    Convert pixel strings to numpy arrays and normalize.
    
    Args:
        df: DataFrame with 'pixels' and 'emotion' columns
        
    Returns:
        X: Normalized image arrays (samples, 48, 48, 1)
        y: Emotion labels (samples,)
    """
    # Extract pixel strings
    pixels = df['pixels'].tolist()
    
    # Convert to arrays
    X = []
    for pixel_sequence in pixels:
        # Split string and convert to integers
        face = [int(pixel) for pixel in pixel_sequence.split()]
        # Reshape to 48x48 image
        face = np.array(face).reshape(48, 48)
        X.append(face)
    
    X = np.array(X)
    
    # Normalize to 0-1 range (pixels are 0-255)
    X = X.astype('float32') / 255.0
    
    # Add channel dimension for CNN: (samples, 48, 48, 1)
    X = X.reshape(-1, 48, 48, 1)
    
    # Get labels
    y = df['emotion'].values
    
    return X, y

print("Preprocessing function defined successfully!")

## 6. Split Dataset (Train/Validation/Test)

In [ ]:
# Split by Usage column
train_data = train_df[train_df['Usage'] == 'Training']
test_data = train_df[train_df['Usage'] == 'PublicTest']

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# Prepare features and labels
print("\nPreparing training data...")
X_train, y_train = prepare_data(train_data)

print("Preparing test data...")
X_test, y_test = prepare_data(test_data)

# Create validation split from training data (20%)
print("Creating validation split...")
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42,
    stratify=y_train  # Keep same emotion distribution
)

print("\n" + "="*50)
print("FINAL DATA SPLIT")
print("="*50)
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")
print("="*50)

## 7. Build CNN Model

In [ ]:
def create_model():
    """
    Create a simple CNN for emotion recognition.
    Architecture: 3 Conv blocks + Flatten + 2 Dense layers
    """
    model = Sequential([
        # Block 1: Extract basic features (edges, textures)
        Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1), name='conv1'),
        BatchNormalization(name='bn1'),
        MaxPooling2D((2, 2), name='pool1'),  # 48x48 -> 24x24
        Dropout(0.25, name='dropout1'),
        
        # Block 2: Detect patterns (eyes, nose, mouth)
        Conv2D(64, (3, 3), activation='relu', name='conv2'),
        BatchNormalization(name='bn2'),
        MaxPooling2D((2, 2), name='pool2'),  # 24x24 -> 12x12
        Dropout(0.25, name='dropout2'),
        
        # Block 3: Capture high-level features (expressions)
        Conv2D(128, (3, 3), activation='relu', name='conv3'),
        BatchNormalization(name='bn3'),
        MaxPooling2D((2, 2), name='pool3'),  # 12x12 -> 6x6
        Dropout(0.25, name='dropout3'),
        
        # Classification layers
        Flatten(name='flatten'),
        Dense(256, activation='relu', name='fc1'),
        BatchNormalization(name='bn4'),
        Dropout(0.5, name='dropout4'),
        Dense(7, activation='softmax', name='output')  # 7 emotions
    ], name='EmotionCNN')
    
    return model

# Create model
model = create_model()

# Display architecture
model.summary()

## 8. Compile Model

In [ ]:
# Compile model with optimizer, loss, and metrics
model.compile(
    optimizer='adam',                          # Adaptive learning rate optimizer
    loss='sparse_categorical_crossentropy',    # For integer labels (0-6)
    metrics=['accuracy']                       # Track accuracy during training
)

print("Model compiled successfully!")
print("\nOptimizer: Adam")
print("Loss: Sparse Categorical Crossentropy")
print("Metrics: Accuracy")

## 9. Setup Training Callbacks

In [ ]:
# Define callbacks to improve training
callbacks = [
    # Stop training if validation loss doesn't improve for 5 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate if validation loss plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # Reduce LR by half
        patience=3,           # Wait 3 epochs before reducing
        min_lr=1e-7,          # Minimum learning rate
        verbose=1
    )
]

print("Training callbacks configured:")
print("- Early Stopping (patience=5)")
print("- Learning Rate Reduction (patience=3)")

## 10. Train Model

In [ ]:
print("="*60)
print("STARTING TRAINING...")
print("="*60)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,                 # Maximum epochs
    batch_size=64,             # Process 64 images at a time
    callbacks=callbacks,
    verbose=1                  # Show progress bar
)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

## 11. Evaluate Model

In [ ]:
# Evaluate on all datasets
print("="*60)
print("MODEL EVALUATION")
print("="*60)

train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"\nTraining Accuracy:   {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")
print(f"Test Accuracy:       {test_acc*100:.2f}%")

print(f"\nTraining Loss:       {train_loss:.4f}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Test Loss:           {test_loss:.4f}")
print("="*60)

## 12. Visualize Training History

In [ ]:
# Plot training history
plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Visualize Predictions

In [ ]:
def show_predictions(X, y, num_images=10):
    """
    Display sample predictions with confidence scores.
    Green border = correct, Red border = incorrect
    """
    # Randomly select images
    indices = np.random.choice(len(X), num_images, replace=False)
    
    plt.figure(figsize=(16, 7))
    
    for i, idx in enumerate(indices):
        # Get image and true label
        img = X[idx].reshape(48, 48)
        true_label = y[idx]
        
        # Make prediction
        pred = model.predict(X[idx:idx+1], verbose=0)
        pred_label = np.argmax(pred)
        confidence = pred[0][pred_label] * 100
        
        # Plot
        plt.subplot(2, 5, i+1)
        plt.imshow(img, cmap='gray')
        
        # Color border based on correctness
        color = 'green' if pred_label == true_label else 'red'
        
        plt.title(
            f'True: {emotions[true_label]}\nPred: {emotions[pred_label]}\n({confidence:.1f}%)', 
            color=color, 
            fontsize=10,
            fontweight='bold'
        )
        plt.axis('off')
    
    plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Show predictions on test set
show_predictions(X_test, y_test, num_images=10)

## 14. Confusion Matrix

In [ ]:
# Generate predictions for all test samples
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotions, yticklabels=emotions,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=13)
plt.xlabel('Predicted Label', fontsize=13)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Diagonal values (dark blue) = correct predictions")
print("- Off-diagonal values = misclassifications")

## 15. Classification Report

In [ ]:
# Detailed classification metrics
print("="*70)
print("CLASSIFICATION REPORT - TEST SET")
print("="*70)
print(classification_report(y_test, y_pred, target_names=emotions))
print("="*70)

print("\nMetrics Explanation:")
print("- Precision: Of all predicted X, how many were actually X?")
print("- Recall: Of all actual X, how many did we predict as X?")
print("- F1-score: Harmonic mean of precision and recall")
print("- Support: Number of actual occurrences in test set")

## 16. Per-Emotion Accuracy

In [ ]:
# Calculate accuracy per emotion
emotion_accuracy = {}

for i, emotion in enumerate(emotions):
    # Find indices where true label is this emotion
    mask = y_test == i
    
    # Calculate accuracy for this emotion
    if mask.sum() > 0:
        correct = (y_pred[mask] == i).sum()
        total = mask.sum()
        accuracy = (correct / total) * 100
        emotion_accuracy[emotion] = accuracy

# Plot per-emotion accuracy
plt.figure(figsize=(12, 6))
colors = ['#e74c3c' if acc < 50 else '#3498db' if acc < 60 else '#2ecc71' 
          for acc in emotion_accuracy.values()]
plt.bar(emotion_accuracy.keys(), emotion_accuracy.values(), color=colors, alpha=0.8)
plt.axhline(y=test_acc*100, color='red', linestyle='--', label=f'Overall: {test_acc*100:.1f}%', linewidth=2)
plt.title('Accuracy per Emotion', fontsize=14, fontweight='bold')
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 100)
plt.xticks(rotation=45)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nPer-Emotion Accuracy:")
for emotion, acc in sorted(emotion_accuracy.items(), key=lambda x: x[1], reverse=True):
    print(f"{emotion:10s}: {acc:5.2f}%")

## 17. Save Model

In [ ]:
# Save the trained model
model.save('fer2013_emotion_model.h5')
print("Model saved as 'fer2013_emotion_model.h5'")

# Also save in SavedModel format (recommended for deployment)
model.save('fer2013_emotion_model_savedformat')
print("Model saved in SavedModel format")

print("\nTo load the model later:")
print("model = keras.models.load_model('fer2013_emotion_model.h5')")

## 18. Test on Single Image (Optional)

In [ ]:
def predict_emotion(image_index):
    """
    Predict emotion for a single test image and show all probabilities.
    """
    # Get image
    img = X_test[image_index]
    true_label = y_test[image_index]
    
    # Predict
    pred = model.predict(img.reshape(1, 48, 48, 1), verbose=0)
    pred_label = np.argmax(pred)
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Show image
    ax1.imshow(img.reshape(48, 48), cmap='gray')
    ax1.set_title(f'True: {emotions[true_label]}', fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Show probabilities
    colors = ['green' if i == pred_label else 'skyblue' for i in range(7)]
    ax2.barh(emotions, pred[0] * 100, color=colors, alpha=0.8)
    ax2.set_xlabel('Probability (%)', fontsize=12)
    ax2.set_title(f'Prediction: {emotions[pred_label]} ({pred[0][pred_label]*100:.1f}%)', 
                  fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAll Probabilities:")
    for i, emotion in enumerate(emotions):
        print(f"{emotion:10s}: {pred[0][i]*100:5.2f}%")

# Test on a random image
random_idx = np.random.randint(0, len(X_test))
predict_emotion(random_idx)

## 19. Summary & Next Steps

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)
print(f"\nDataset: FER2013")
print(f"Task: 7-class emotion classification")
print(f"Model: Simple CNN (3 conv blocks)")
print(f"\nFinal Results:")
print(f"  Training Accuracy:   {train_acc*100:.2f}%")
print(f"  Validation Accuracy: {val_acc*100:.2f}%")
print(f"  Test Accuracy:       {test_acc*100:.2f}%")
print(f"\nExpected Range: 55-65% (baseline CNN)")
print("="*70)

print("\nWAYS TO IMPROVE:")
print("1. Data Augmentation (rotation, flip, zoom)")
print("2. Transfer Learning (VGG16, ResNet, EfficientNet)")
print("3. Deeper CNN architecture")
print("4. Ensemble methods (combine multiple models)")
print("5. Attention mechanisms")
print("6. Class balancing (handle imbalanced emotions)")
print("7. Learning rate scheduling")
print("8. Mixup / CutMix augmentation")
print("\nNotebook complete!")